In [ ]:
import os
import sys
import re
import math 


import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq
)
from tqdm.auto import tqdm
from sacrebleu.metrics import CHRF
import transformers

print(transformers.__version__)

5.2.0


# Inference with the PY file

## Configurations


In [ ]:
config = {
    # Model paths
    "model_path": "/kaggle/input/models/analeopold/15kext-dict-intern-3-1-6weights/pytorch/default/1/checkpoint-63540", # Best performing ByT5 model
    "test_data_path": "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv", 
    "max_length":512,
    "batch_size":8, 
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    
    # Text processing arguments
    "separate_compounds": False,
    "with_hyphens": False,
    "named_determinatives": True,
    "normalize_chars": True,

    # OTHER EXTRA parameters 
    "num_workers": 4,
    "length_penalty": 1, # means no penalty
    "early_stopping": False,
    "no_repeat_ngram_size": 0,
    "use_bucket_batching": True, 
    "num_buckets": 8,  
    "num_beams": 8,
    "mixed_precision": False, 

    # MBR Parameters
    "max_new_tokens": 256, # Maximum tokens to be generated, used for (expensive) MBR generation
    "mbr": False,
    "mbr_mode": "hybrid", # sample, beam, hybrid
    
    "mbr_samples": 4, 
    "mbr_beams": 4,
    
    "mbr_temperature": 0.7, # Temperature when using sample-based generation
    "mbr_pruning": False, # Pruning number of candidates for MBR using HF generation scores
    "mbr_keep_k": 6 # Number of candidates to keep for MBR after pruning 
    }

## Processing Class

In [83]:
class CompiledPatterns:
    """Pre-compile all regex patterns for pre and post processing"""


    ################################################################################
    ################################################################################
    ##                                                                            ##
    ##                            AKKADIAN WRITING                                ##
    ##                                                                            ##
    ################################################################################
    ################################################################################

    # Logograms as fully capitalized words
    CHARS_CAPITAL = r"[A-ZŠŠṢṢṬṬḪḪÁÉÍÚÀÈÌÙ]"

    LOGOGRAM_STR = fr"(?!(?:PN)\b){CHARS_CAPITAL}+(?:[\.·]{CHARS_CAPITAL}+)*"

    LOGOGRAM = re.compile(fr"\b({LOGOGRAM_STR})\b")
    LOGOGRAMS_COMPOUND = re.compile(fr"\b({LOGOGRAM_STR})[\.·]({LOGOGRAM_STR})\b")
    
    ALL_CHARS = r"a-zA-Z0-9šŠṣṢṭṬḫḪáéíúàèìù<>,·\."
    HYPHENS = re.compile(fr"(?<=[{ALL_CHARS}])\-(?=[{ALL_CHARS}])")


    # Global translation tables
    # TODO: i dont understand what we should do
    ACCENT_MAP = {
        "á": "a2", "é": "e2", "í": "i2", "ú": "u2",
        "Á": "A2", "É": "E2", "Í": "I2", "Ú": "U2",
        
        "à": "a3", "è": "e3", "ì": "i3", "ù": "u3",
        "À": "A3", "È": "E3", "Ì": "I3", "Ù": "U3",
        
        "š": "sz", "Š": "SZ",
        "ṣ": "s,", "Ṣ": "S,",
        "ṭ": "t,", "Ṭ": "T,",
        "ḫ": "h", "Ḫ": "H",
    }
    
    NORM_PATTERN = re.compile("|".join(re.escape(k) for k in sorted(ACCENT_MAP.keys(), key=len, reverse=True)))

    SUBSCRIPT_TABLE = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")

    # matches normalized and lower case form
    DETERMINATIVE_MAP = {
        "{d}": " god",
        "{mul}": " stars",
        "{ki}": " earth",
        "{lú}": " person",      
        "{é}": " building",    
        "{uru}": " settlement",
        "{kur}": " land",
        "{munus}": " feminine", 
        "{m}": " masculine",
        "{giš}": " wood",      
        "TUG": " textile",
        "{dub}": " tablet",
        "{id2}": " river",
        "{mušen}": " bird",
        "{na4}": " stone",
        "{kuš}": " skin",
        "{ú}": " plant",
    }

    DETERMINATIVE_PATTERN = re.compile("|".join(re.escape(k) for k in sorted(DETERMINATIVE_MAP.keys(), key=len, reverse=True)))

    # fix host said idk why
    # KÙ.B. —> KÙ.BABBAR
    KUB_PATTERN = re.compile(r"k[ùu]3?\.b\.", re.I)

    
    ################################################################################
    ################################################################################
    ##                                                                            ##
    ##                                   GAPS                                     ##
    ##                                                                            ##
    ################################################################################
    ################################################################################

    # Input preprocessing
    # Matches: x, [x], …, (break), (large break), (n broken lines)    # will be replaced with <gap>
    # probably useless 
    GAP_INPUT = re.compile(
        r"(PN|\[x\]|\(x\)|\(\d*\s*broken\s+lines?\b\s*\)|\.{3,}|…+|\bxx+\b|\bx\b)", 
        re.I
    )
    # Intransliteration looks for: ..., …, [... ...], or [ . . . ]: unknown number of missing signs/lines
    # will bed with <big_gap>
    # REMOVED IN NEW VERSION moved to <gap> for all 
    # BIG_GAP_INPUT = re.compile(r"(\[x\]|\(x\)|(?<!\w)x(?!\w)|xx+|\s+x\s+)")
    
    # Output postprocessing
    # MERGE_GAPS: Merges sequences consisting only of <gap>
    # Matches: <gap> <gap>, -<gap> <gap>, <gap>-<gap>, etc.
    MERGE_GAPS = re.compile(r"-?<gap>(?:\s*[\-\s]*<gap>)+-?")

    # MERGE_BIG_GAPS: Merges sequences consisting only of <big_gap>
    # Matches: <big_gap> <big_gap>, -<big_gap> <big_gap>-, etc.
    MERGE_BIG_GAPS = re.compile(r"-?<big_gap>(?:\s*[\-\s]*<big_gap>)+-?")

    # MERGE_MIXED_GAPS: Merges sequences containing both <gap> and <big_gap>
    # Matches: <gap> <big_gap>, -<big_gap> <gap> <big_gap>-, etc.
    MERGE_MIXED_GAPS = re.compile(r"-?(?:<gap>|<big_gap>)(?:\s*[\-\s]*(?:<gap>|<big_gap>))+-?")

    # GRAMMAR_ANNOTATIONS = re.compile(
    #     r"\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)", re.I
    # )

    
    ################################################################################
    ################################################################################
    ##                                                                            ##
    ##                                CLANUPS                                     ##
    ##                                                                            ##
    ################################################################################
    ################################################################################

    # post processing cleanup
    REPEATED_WORDS = re.compile(r"\b(\w+)(?:\s+\1\b)+")
    SPACE_BEFORE_PUNCT = re.compile(r"\s+([.,:])")
    REPEATED_PUNCT = re.compile(r"([.,])\1+")
    MULTI_SPACE = re.compile(r"\s+")

    # N-gram dedup (pre-compiled for sizes 4,3,2)
    NGRAM_PATTERNS = [
        re.compile(r"\b((?:\w+\W+){" + str(n - 1) + r"}\w+)(?:\W+\1\b)+")
        for n in range(6, 1, -1)
    ]

    BRACKETS_MAP = {'{': '(', '}': ')'} # not used anymore after dataset change
    BRACKETS_PATTERN = re.compile(r'[{}]')

    FORBIDDEN_CHARS_INPUT = str.maketrans("", "", "\"!?()—–⌈⌋⌊+'/;")
    FORBIDDEN_CHARS_OUTPUT = str.maketrans("", "", "()—–⌈⌋⌊+/;")

    # TODO how to remove "not meaningful ? and !"... agh
    REAL_QUESTION_PATTERN = re.compile(r"\b(Who|What|Where|When|Why|How|Whose|Which)\b([^.!?]*)(\?)", re.I)

    ENG_STRAY_CLEANUP = re.compile(
        r"("
        # Matches (fem.), (fem), fem., or fem
        r"\(?\b(fem|sing|plur|plural|pl|masc)\b\.?\)?|" 
        r"\(\?\)|"                             # Matches (?)
        r"\.{2,}|"                             # Matches .. or ...
        r"\?|"                                 # Matches standalone ?
        r"\bxx?\b|"                            # Matches standalone x or xx
        r"<<|>>|"                              # Matches << or >>
        r"<(?!/?(?:big_)?gap>)|"               # Matches < (not part of <gap>)
        r"(?<!gap)>"                           # Matches > (not part of <gap>)
        r")",
        re.I
    )

    ENG_SLASH_CHOICE = re.compile(r"(\b[\w\s'-]+)\s*/\s*([\w\s'-]+\b)")

    WORD_EXPANSIONS = [
        # Match '-gold' only if it's not preceded by another letter
        # This keeps 'gold' as is, but changes '-gold'
        (re.compile(r"(?<!\w)-gold\b", re.I), "pašallum gold"),
        
        # Match '-tax' only if it's not preceded by another letter
        (re.compile(r"(?<!\w)-tax\b", re.I), "šadduātum tax"),
        
        # Match 'textile' or 'textiles'
        (re.compile(r"(?<!\w)-textiles?\b", re.I), "kutānum textiles"),
    ]

    
    ################################################################################
    ################################################################################
    ##                                                                            ##
    ##                            NUMBERS AND SUCH                                ##
    ##                                                                            ##
    ################################################################################
    ################################################################################

    FRACTION_CONVERSION = {
        "½": re.compile(r"\b(0\.5|one[- ]half)\b", re.I),
        "¼": re.compile(r"\b(0\.25|one[- ]fourth|one[- ]quarter)\b", re.I),
        "⅓": re.compile(r"\b(0\.33\d*|one[- ]third)\b", re.I),
        "⅚": re.compile(r"\b(0\.83\d*|five[- ]sixths)\b", re.I),
        "⅝": re.compile(r"\b(0\.625|five[- ]eighths)\b", re.I),
        "⅔": re.compile(r"\b(0\.66\d*|0\.67|two[- ]thirds)\b", re.I),
        "¾": re.compile(r"\b(0\.75|three[- ]fourths|three[- ]quarters)\b", re.I),
        "⅙": re.compile(r"\b(0\.16\d*|0\.17|one[- ]sixth)\b", re.I),
    }
      
    NUMBERS_MAP = {
        "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
        "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
        "eleven": "11", "twelve": "12", "thirteen": "13", "fourteen": "14", "fifteen": "15",
        "sixteen": "16", "seventeen": "17", "eighteen": "18", "nineteen": "19", "twenty": "20",
        "thirty": "30", "forty": "40", "fifty": "50", "sixty": "60",
        "seventy": "70", "eighty": "80", "ninety": "90", "hundred": "100"
    }

    # Regex for "one" with exceptions:
    # Don't match if preceded by 'some', 'any', 'no', 'every', 'each', 'at'
    # Don't match if followed by 'another'
    # are there other issues with one?
    ONE_CLEAN_REGEX = (
            r"(?<!some\s)(?<!any\s)(?<!no\s)(?<!every\s)(?<!each\s)(?<!at\s)"
            r"\bone\b"
            r"(?!\s+another)"
        )
    other_nums = "|".join([k for k in NUMBERS_MAP.keys() if k != "one"])
    OTHER_NUMBERS_PATTER = re.compile(rf"\b({other_nums})\b", re.I)

    # roman numbers 
    MONTH_NUMERAL_REGEX = re.compile(
        r"\b(Month|Months?|Nisannu|Ajaru|Simanu|Abu|Ululu|Taszritu|Arahsamnu|Kislimu|Tebetu|Szabatu|Addaru)\s+([IVX]+)\b", 
        re.I
    )

    # Simple map for the 12 months
    ROMAN_MAP = {
        'I': '1', 'II': '2', 'III': '3', 'IV': '4', 'V': '5',
        'VI': '6', 'VII': '7', 'VIII': '8', 'IX': '9', 'X': '10',
        'XI': '11', 'XII': '12'
    }

    # DO NOT change order plzz: Longest/most specific patterns first
    MONEY_PATTERNS = [
        # 5 11/12 shekels
        (re.compile(r"5\s+11\s*/\s*12\s*shekels?", re.I), "6 shekels less 15 grains"),
        
        # 1/12 shekel (handles: 1/12 shekel, 1/12 shekels, 1/12 (shekel), etc.)
        (re.compile(r"1\s*/\s*12\s*\(?shekels?\)?", re.I), "⅔ shekel 15 grains"),
        
        # 5/12 shekel
        (re.compile(r"5\s*/\s*12\s*\(?shekels?\)?", re.I), "15 grains"),
        
        # 7/12 shekel
        (re.compile(r"7\s*/\s*12\s*\(?shekels?\)?", re.I), "½ shekel 15 grains"),
    ]


# BASE_DIR = os.path.dirname(__file__)
# BASE_DIR = os.path.join(BASE_DIR, "..", "..", "dataset")

# PREPROCESSING_CONFIG = {
#     "test_data_path": os.path.join(BASE_DIR, "test.csv"),
#     "train_data_path": os.path.join(BASE_DIR, "train.csv"),
#     "computation_device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
# }


class TextProcessor:
    """Provide pre and post processing functions"""

    def __init__(self):
        self.patterns = CompiledPatterns()

    def _handle_determinatives(self, match):
        """replace or leave intact determinatives and log to wandb if missing"""
        det = match.group(0)
        
        replacement = self.patterns.DETERMINATIVE_MAP.get(det, None)
        
        if replacement:
            # Surround with spaces so it doesn't stick to the word
            return f" {replacement} "
        
        return det


    def preprocess_transliteration_text(
        self,
        transliteration_text,
        separate_compounds: bool = False,
        with_hyphens: bool = False,
        named_determinatives: bool = False,
        normalize_chars: bool = True
    ):
        """Preprocessess transliteration text

        Args:
            transliteration_text: text to be preprocessed
            separate_compounds (bool, optional): if true separates logograms compounds into two space-separated words.
                Defaults to False.
            with_hyphens (bool, optional): if true maintains hypheneted words, else removes hyphens and joins syllabi
                (e.g. a-na -> ana). Defaults to False.
            named_determinatives (bool, optional): if true substitute determiantives with meaningful word,
                else leave intact. Defaults to False.

        Returns:
            text: preprocessed text

        Notes:
            preprocessing steps:
            - changes subscripts, superscripts and accents to unifified letters (e.g. "á" to "a2")
            - Add special token to indicate start (<lgg_s>) and end (<lgg_e>) of logograms
            - Lowercase everything
            - change remaining subscripts to normal numbers
            - change gaps to <gap> and <big_gap>
            - handle determinatives
            - optionally remove hyphens
            - optionally separate logogram compounds with spaces
            - remove forbidden characters
            - remove multispaces
        """
        if pd.isna(transliteration_text):
            return ""

        text = str(transliteration_text)

        # idk why host said this
        text = self.patterns.KUB_PATTERN.sub("KÙ.BABBAR", text)

        # normalize gaps
        # text = self.patterns.BIG_GAP_INPUT.sub("<big_gap>", text)
        # BEFORE logograms or problem with PN
        text = self.patterns.GAP_INPUT.sub("<gap>", text)
        text = self.patterns.MERGE_GAPS.sub(" <gap> ", text)
        text = self.patterns.MERGE_BIG_GAPS.sub(" <gap> ", text)
        text = self.patterns.MERGE_MIXED_GAPS.sub(" <gap> ", text)

        # OPTIONAL: separate logograms compound with space
        if separate_compounds:
            text = self.patterns.LOGOGRAMS_COMPOUND.sub(r"\1 \2", text)
        # add special tokens for logograms start and end, and lowecase
        text = self.patterns.LOGOGRAM.sub(lambda m: f"<lgg_s>{m.group(1)}<lgg_e>", text)

        # OPTIONAL: sub determinatives with word
        if named_determinatives:
            # replace if found, leave and log if not found
            text = self.patterns.DETERMINATIVE_PATTERN.sub(self._handle_determinatives, text)

        # normalize characters
                # normalize subscripts
        if normalize_chars:
            text = text.translate(self.patterns.SUBSCRIPT_TABLE)

        if normalize_chars:
            for acc, num in self.patterns.ACCENT_MAP.items():
                text = text.replace(acc, num)

        text = text.lower()

        # OPTIONAL: remove hyphens
        if not with_hyphens:
            text = self.patterns.HYPHENS.sub("", text)

        text = text.translate(self.patterns.FORBIDDEN_CHARS_INPUT)

        text = self.patterns.MULTI_SPACE.sub(" ", text).strip()

        return text
    
    def preprocess_translation_text(self, translation_text):
        if pd.isna(translation_text): return ""
        text = str(translation_text)

        text = text.replace(fr"\(?\)", "")
        # protect real questions
        text = self.patterns.REAL_QUESTION_PATTERN.sub(r"\1\2REALQ", text)

        for pattern, replacement in self.patterns.MONEY_PATTERNS:
            text = pattern.sub(replacement, text)

        # change month+roman to month+digit
        text = self.patterns.MONTH_NUMERAL_REGEX.sub(
            lambda m: f"{m.group(1)} {self.patterns.ROMAN_MAP.get(m.group(2).upper(), m.group(2))}", 
            text
        )

        text = self.patterns.GAP_INPUT.sub("<gap>", text)

        for symbol, pattern in self.patterns.FRACTION_CONVERSION.items():
            text = pattern.sub(f" {symbol}", text)

        text = re.sub(self.patterns.ONE_CLEAN_REGEX, "1", text, flags=re.I)
        text = self.patterns.OTHER_NUMBERS_PATTER.sub(
            lambda m: self.patterns.NUMBERS_MAP[m.group(0).lower()], text
        )

        for pattern, replacement in self.patterns.WORD_EXPANSIONS:
            text = pattern.sub(replacement, text)
        
        # clean up 
        text = self.patterns.ENG_SLASH_CHOICE.sub(r"\1", text) 
        text = self.patterns.ENG_STRAY_CLEANUP.sub("", text)
        text = text.translate(self.patterns.FORBIDDEN_CHARS_OUTPUT)

        # restore real questions
        text = text.replace("REALQ", "?")

        text = self.patterns.MULTI_SPACE.sub(" ", text).strip()

        return text

    def postprocess_translation_output(self, translation_text):
        """Postprocess translation text

        Args:
            translation_text: translated text

        Returns:
            text: processed tranlation text

        Notes:
            postprocessing steps:
            - clean and merge gaps
            - remove repeated words and n-grams (common hallucination)
            - remove extra spaces and punctuations
            - change all text numbers to digits (e.g. one to 1)
            - change all fractions to decimals (e.g. one third to 0.33333)
        """
        if not isinstance(translation_text, str) or not translation_text.strip():
            return ""

        text = str(translation_text)

        # substitute gaps and merge multiple gaps
        text = self.patterns.MERGE_GAPS.sub(" <gap> ", text)
        text = self.patterns.MERGE_BIG_GAPS.sub(" <gap> ", text)
        text = self.patterns.MERGE_MIXED_GAPS.sub(" <gap> ", text)

        # remove repeated words and ngrams (up to 4)
        text = self.patterns.REPEATED_WORDS.sub(r"\1", text)
        for ngram_pattern in self.patterns.NGRAM_PATTERNS:
            text = ngram_pattern.sub(r"\1", text)

        # remove extra spaces
        text = self.patterns.SPACE_BEFORE_PUNCT.sub(r"\1", text)
        text = self.patterns.REPEATED_PUNCT.sub(r"\1", text)
        text = self.patterns.MULTI_SPACE.sub(" ", text).strip().strip("-").strip()

        text = self.patterns.MONTH_NUMERAL_REGEX.sub(
            lambda m: f"{m.group(1)} {self.patterns.ROMAN_MAP.get(m.group(2).upper(), m.group(2))}", 
            text
        )

        # change numbers to digints (one -> 1) from 1 to 10
        text = re.sub(self.patterns.ONE_CLEAN_REGEX, "1", text, flags=re.I)

        # Process all other numbers (two through hundred)
        text = self.patterns.OTHER_NUMBERS_PATTER.sub(
            lambda m: self.patterns.NUMBERS_MAP[m.group(0).lower()], text
        )

        # change fractions to decimals
        for symbol, pattern in self.patterns.FRACTION_CONVERSION.items():
            text = pattern.sub(symbol, text)

        return text

## DataSet Class

In [ ]:
class AkkadianTranslationDatasetT5(Dataset):
    """
    This class creates a dataset depending on the mode (train vs inference) and
    the task (translation vs reconstruction).

    Input requires already preprocessed data, meaning it should come after
    running TextProcessor() and the corresponding function on the text.
    Tokenizer needs to be initialized beforehand. It can be used like this:

    train_dataset = AkkadianTranslationDatasetT5(train_df,tokenizer)

    """

    def __init__(
        self, dataframe, tokenizer, max_length=512, mode="train", task="translation"
    ):
        """
        Intializes the dataset with a dataframe. Depending on the task, the
        input and target texts are created differently.

        Args:
            dataframe (pd.DataFrame): The dataframe with the train, validation
            or test data, already preprocessed. tokenizer (PreTrainedTokenize):
            The HuggingFace tokenizer from the pretrained model. max_length
            (int, optional): Sequence truncation length. Defaults to 512. mode
            (str, optional): "train" or "inference" Defaults to "train". task
            (str, optional): "translation" or "reconstruction". Defaults to
            "translation".
        """
        # self.sample_ids = dataframe["id"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode
        self.task = task  # when denoising mode is true, the task is reconstruction rather than translation

        if task == "translation":
            # Input: transliteration with translation 
            self.input_texts = [
                "translate Akkadian to English: " + str(t)
                for t in dataframe["transliteration"]
            ]
            if self.mode == "train":
                # Target: English translation
                self.target_texts = [str(text) for text in dataframe["translation"]]
        else:
            # Input: Noisy transliteration with reconstruction prompt
            self.input_texts = [
                "reconstruct Akkadian from the noise: " + str(t)
                for t in dataframe["noisy_transliteration"]
            ]
            # Target: original transliteration
            self.target_texts = [str(text) for text in dataframe["transliteration"]]


        # Keep original order for bucket batching
        self.indices = list(range(len(self.input_texts)))

        # Precompute tokenized input lengths for bucket batching
        self.input_lengths = [
            min(
                len(
                    self.tokenizer(
                        text,
                        truncation=True,
                        max_length=self.max_length,
                        add_special_tokens=True
                    )["input_ids"]
                ),
                self.max_length 
            )
            for text in self.input_texts
        ]

    
    def __len__(self):
        return len(self.input_texts)

    def __getitem__(self, index):
        """
        Retrieves and tokenizes a single data sample by its index. Processes the
        input text (transliteration) and target text (translation or original
        transliteration) into tensors. If in "train" mode, it includes labels.

        Args:
            index (int): Index of the data sample to retrieve.

        Returns:
            dict: A dictionary containing:
                - "input_ids": Tokenized input text as a tensor.
                - "attention_mask": Attention mask for the input text as a tensor.
                - "labels" (optional): Tokenized target text as a tensor, included only in "train" mode.
        """
        # Tokenize transliteration text (either normal or noisy)
        tokenized_inputs = self.tokenizer(
            self.input_texts[index],
            padding=False, # this happens in data collator
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        )

        # Base model inputs
        item = {
            "input_ids": tokenized_inputs["input_ids"].squeeze(),
            "attention_mask": tokenized_inputs["attention_mask"].squeeze(),
            # Added for Bucket Batching
            "idx": self.indices[index], # for getting order
        }

        # Add target labels during training
        if self.mode == "train":
            tokenized_target = self.tokenizer(
                self.target_texts[index],
                padding=False, # this happens in data collator
                max_length=self.max_length,
                truncation=True,
                return_tensors="pt",
            )
            
            labels = tokenized_target["input_ids"].squeeze()
            labels[labels == self.tokenizer.pad_token_id] = -100
            item["labels"] = labels

        return item

## Buckets Batch Sampling
Inspired by Gregory L., who copied the notebook from Mattia Angeli: https://www.kaggle.com/code/lgregory/akkadiam-exemple

In [85]:

class BucketBatchSampler(Sampler):
    """
    Custom PyTorch sampler that groups sequences of similar lengths 
    into the same batch (bucket batching). 

    Args:
        lengths (list[int]): Tokenized input length for each sample in the dataset.
        batch_size (int): Number of samples per batch.
        num_buckets (int, optional): Number of length-based buckets.
        drop_last (bool, optional): Drop final batch in a bucket if smaller than batch_size. 
    """
    def __init__(self,lengths, batch_size, num_buckets=4, drop_last=False):
        self.lengths = lengths
        self.batch_size = batch_size
        self.num_buckets = num_buckets
        self.drop_last = drop_last

        # Sort indices by sequence length
        sorted_ids = sorted(range(len(lengths)), key=lambda i: lengths[i])

        bucket_size = max(1, len(sorted_ids) // max(1, num_buckets))
        
        self.batches = []

        # Looping over buckets, adding buckets to batches
        for i in range(num_buckets):
            start = i*bucket_size
            end = min((i+1)*bucket_size, len(sorted_ids))
            bucket = sorted_ids[start:end]

            # Creating batches from buckets (skip with batch_size to fill batches)
            for j in range(0,len(bucket), batch_size):
                batch = bucket[j:j+batch_size]
                if len(batch)==batch_size or not drop_last:
                    self.batches.append(batch)

        # Collect any remaing samples not yet covered 
        remainder_start = num_buckets*bucket_size
        if remainder_start < len(sorted_ids):
            bucket = sorted_ids[remainder_start:]
            for j in range(0, len(bucket), batch_size):
                batch = bucket[j:j+batch_size]
                if len(batch) == batch_size or not drop_last:
                    self.batches.append(batch)

    def __iter__(self):
        yield from self.batches

    def __len__(self):
        return len(self.batches)
                

## Collator for bucket batching
Wrapping DataCollatorForSeq2Seq to retain ids for bucket batching

In [86]:
class InferenceCollator:
    """
    Wraps DataCollatorSeq2Seq to preserve sample indices.
    Idx field is popped before collation and reattached after.

    Args:
        tokenizer: HuggingFace tokenizer used for padding.
        model: Model used in inference.
    """
    def __init__(self,tokenizer, model=None):
        self.base_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    def __call__(self, features):
        # Remove idx before collation
        ids = [f.pop("idx") for f in features]
        batch = self.base_collator(features)

        # Reattach idx
        batch["idx"] = torch.tensor(ids, dtype=torch.long)
        return batch

# Inference class 

Modularized, with MBR 

In [ ]:
# Inference.py

import torch
import tqdm
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq
import pandas as pd
from torch.utils.data import DataLoader

class byT5Inference:
    """
    Inference class for byT5 Akkadian translation
    All configurations defined directly in Kaggle notebook
    """
    def __init__(self, config):
        """
        Args:
            config (dict): Configuration dictionary defined in Kaggle notebook
        """
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config["model_path"],
                                                           local_files_only=True)
        try:
            self.model = AutoModelForSeq2SeqLM.from_pretrained(config["model_path"], 
                                                           attn_implementation="sdpa",
                                                            local_files_only=True).to(config["device"])
        except ValueError:
            print("SDPA not supported, falling back to default")
            self.model = AutoModelForSeq2SeqLM.from_pretrained(config["model_path"], 
                                                               local_files_only=True).to(config["device"])
            
        self.processor = TextProcessor()
        if self.config["mbr"]:
            self.mbr_metric = CHRF(word_order=2)
            
    def prepare_dataloader(self):
        """
        Loads and preprocesses the test CSV, returns ready dataset
        """
        data = pd.read_csv(self.config["test_data_path"])
        
        data["transliteration"] = data["transliteration"].apply(
            lambda x: self.processor.preprocess_transliteration_text(
                   transliteration_text=x,
                   separate_compounds=self.config["separate_compounds"],
                   with_hyphens=self.config["with_hyphens"], 
                   named_determinatives=self.config["named_determinatives"]))
        
        test_dataset = AkkadianTranslationDatasetT5(data, self.tokenizer, max_length=self.config["max_length"], mode="inference")
        
        return test_dataset


    def mbr_select(self, candidates):
        """
        Picks best candidate via MBR. Builds full similarity matrix and returns
        candidate with highest average CHRF similarity to all other candidates.

        Args:
            candidates (list[str]): Candidate strings from generation

        Returns:
            str: Best candidate or empty string if there are no valid candidates 
        """
        # Strip whitespaces, drop empty candidates, remove duplicates
        candidates = list(dict.fromkeys(c.strip() for c in candidates if c.strip()))
        
        n = len(candidates)

        if n == 0:
            return ""
        if n==1:
            return candidates[0]
            
        similarity = [[0.0]*n for _ in range (n)]
        
        # Only compute upper triangle, then mirror
        for i in range(n):
            for j in range(i+1,n):
                a = candidates[i]
                b = candidates[j]

                if not a or not b:
                    s = 0.0
                else:
                    # Changed from corpus score to sentence score
                    s = self.mbr_metric.sentence_score(a, [b]).score

                similarity[i][j] = s
                similarity[j][i] = s
        
        best_ids = max(range(n), key=lambda i: sum(similarity[i]))

        return candidates[best_ids]

    def prune_candidates(self, candidates, scores, keep_k=6):
        """
        Keeps top-k candidates before expensive MBR. 

        Args:
            candidates (list[str]): Candidate translations
            scores (list[float]): Huggingface generation scores
            keep_k (int): Number of candidates to keep

        Returns:
            list[str]: Pruned candidate list
        """
        if len(candidates) <= keep_k:
            return candidates
        
        pairs = list(zip(candidates, scores))
        pairs.sort(key=lambda x: float(x[1]), reverse = True)
        pruned = [c for c,_ in pairs[:keep_k]]
        return pruned


    def _base_generate_kwargs(self, attention_mask):
        """
        Generates kwargs for regular (beam-based) inference
        """
        return dict(
            attention_mask=attention_mask,
            max_length=self.config["max_length"],
            use_cache=True,
            no_repeat_ngram_size=self.config["no_repeat_ngram_size"],
            early_stopping=self.config["early_stopping"],
            length_penalty=self.config["length_penalty"]
        )


    def _run_generate(self, input_ids, generate_kwargs):
        """
        Wrapper for running model.generate() with or without mixed precision
        """
        if self.config["mixed_precision"]:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = self.model.generate(input_ids, **generate_kwargs)
        else:
            outputs = self.model.generate(input_ids,**generate_kwargs)
        return outputs

    def _decode_generate_output(self, outputs):
        """
        Decodes output, handling both raw tensors and HF generation objects 
        in the case of pruning
        """
        if hasattr(outputs, "sequences"):
            sequences = outputs.sequences

        else:
            sequences = outputs

        return self.tokenizer.batch_decode(
            sequences,
            skip_special_tokens=True
        )
        
    
    def _get_sequence_scores(self, outputs, fallback_len):
        """
        Extract Huggingface scores if available
        """
        if hasattr(outputs, "sequences_scores") and outputs.sequences_scores is not None:
            return outputs.sequences_scores.detach().cpu().tolist()
                            
        return [0.0]*fallback_len # Fallback: assign all candidates same dummy score



    
    def _generate_sample_candidates(self, input_ids, attention_mask):
        """
        Generates  candidates via sampling for MBR

        Returns:
            tuple: (decoded, scores, n_samples)
        """
        n_samples = self.config.get("mbr_samples", 10)

        kwargs = self._base_generate_kwargs(attention_mask)
        kwargs.update(dict(
            max_new_tokens=self.config["max_new_tokens"],
            do_sample=True,
            num_return_sequences=n_samples,
            temperature=self.config.get("mbr_temperature", 0.7),
            num_beams=1,  
            return_dict_in_generate=True, 
            output_scores=True, # For pruning
        ))

        outputs = self._run_generate(input_ids, kwargs)
        decoded = self._decode_generate_output(outputs)
        scores = self._get_sequence_scores(outputs, len(decoded)) 

        return decoded, scores, n_samples


    def _generate_beam_candidates(self, input_ids, attention_mask):
        """
        Generates beam candidates for MBR

        Returns:
            tuple: (decoded, scores, n_samples)
        """
        n_beams = self.config.get("mbr_beams", self.config.get("num_beams", 4))
        n_beams = max(1, n_beams)
        
        kwargs = self._base_generate_kwargs(attention_mask)
        kwargs.update(dict(
            max_new_tokens=self.config["max_new_tokens"],
            do_sample=False,
            num_beams=n_beams,
            num_return_sequences=n_beams,
            return_dict_in_generate=True, 
            output_scores=True, # For pruning
        ))
        
        outputs = self._run_generate(input_ids, kwargs)
        decoded = self._decode_generate_output(outputs)
        scores = self._get_sequence_scores(outputs, len(decoded))     

        return decoded, scores, n_beams

    def _build_mbr_candidate_groups(self, input_ids, attention_mask):
        """
        Returns MBR candidate pools for three modes:
            - "sample": stochastic sampling only
            - "beam": beam search only
            - "hybrid": beam and sample candidates combined

        Returns:
            tuple: (candidates_list, scores_list), one per sample in the batch
        """
        mbr_mode = self.config.get("mbr_mode", "sample")
        batch_size = input_ids.size(0)
        

        if mbr_mode == "sample":
            decoded_all, all_scores, group_size = self._generate_sample_candidates(
                input_ids, attention_mask
            )
            candidates_list = []
            scores_list = []

        elif mbr_mode == "beam":  
            decoded_all, all_scores, group_size = self._generate_beam_candidates(
                input_ids, attention_mask
            )
            candidates_list = []
            scores_list = []
        
        elif mbr_mode == "hybrid":
            sample_decoded, sample_scores, n_samples = self._generate_sample_candidates(
                input_ids, attention_mask
            )

            beam_decoded, beam_scores, n_beams = self._generate_beam_candidates(
                input_ids, attention_mask
            )

            candidates_list = []
            scores_list = []

            for i in range(batch_size):
                sample_start = i * n_samples
                sample_end = (i+1) * n_samples

                beam_start = i * n_beams
                beam_end = (i+1) * n_beams

                candidates = beam_decoded[beam_start:beam_end] + sample_decoded[sample_start:sample_end]
                scores = beam_scores[beam_start:beam_end] + sample_scores[sample_start:sample_end]

                candidates_list.append(candidates)
                scores_list.append(scores)
                
                        
            return candidates_list, scores_list

        
        else:
            raise ValueError(f"Unsupported mbr_mode: {mbr_mode}")

        # for sample and beam modes
        candidates_list = [decoded_all[i * group_size:(i + 1) * group_size] for i in range(batch_size)]
        scores_list = [all_scores[i * group_size:(i + 1) * group_size] for i in range(batch_size)]
        
        return candidates_list, scores_list

    def translate(self, test_dataset):
        """
        Runs inference on test set and returns predictions.
        Supports regular beam search, MBR (sample, beam, hybrid), 
        MBR pruning, and bucket batching.

        Args:
            test_dataset (AkkadianTranslationDatasetT5): Preprocessed dataset from prepare_dataloader()

        Returns:
            list[str]: Predicted translations, in matching order with the test dataset
            
        """

        collator = InferenceCollator(self.tokenizer, model=self.model)

        if self.config.get("use_bucket_batching", False):
            batch_sampler = BucketBatchSampler(
                lengths=test_dataset.input_lengths,
                batch_size=self.config["batch_size"],
                num_buckets=self.config["num_buckets"],
                drop_last=False
            )
            
            test_loader = DataLoader(
                test_dataset, 
                batch_sampler=batch_sampler,
                num_workers=self.config["num_workers"], 
                collate_fn=collator,
                pin_memory=True  
            )
            
        else:
            test_loader = DataLoader(
                test_dataset, 
                batch_size=self.config["batch_size"], 
                num_workers=self.config["num_workers"], 
                shuffle=False,
                collate_fn=collator,
                pin_memory=True
            )


        # Initialising None-filled list to preserve origanl order when adding predictions
        predictions = [None] * len(test_dataset)
        
        with torch.no_grad():
            for batch in tqdm(test_loader):
                batch_indices = batch["idx"].tolist()
                input_ids = batch["input_ids"].to(self.config["device"])
                attention_mask = batch["attention_mask"].to(self.config["device"])
   
                # Standard beam search
                if not self.config["mbr"]:
                    generate_kwargs = self._base_generate_kwargs(attention_mask)
                    generate_kwargs.update(dict(
                        num_beams=self.config["num_beams"]
                    ))

                    outputs = self._run_generate(input_ids, generate_kwargs)
                    decoded = self._decode_generate_output(outputs)

                # MBR Path
                else:
                    keep_k = self.config.get("mbr_keep_k", 6)
                    grouped_candidates, grouped_scores = self._build_mbr_candidate_groups(
                        input_ids, attention_mask
                    )

                    decoded = []
                    for candidates, candidate_scores in zip(grouped_candidates, grouped_scores):
                        if self.config.get("mbr_pruning", False): # Set to False if pruning not specified
                            candidates = self.prune_candidates(
                                candidates,
                                candidate_scores,
                                keep_k=keep_k
                            )

                        # Run MBR metric 
                        best = self.mbr_select(candidates)
                        decoded.append(best)

                post_processed = [
                    self.processor.postprocess_translation_output(text) 
                    for text in decoded
                ]

                for idx, pred in zip(batch_indices, post_processed):
                    predictions[idx] = pred

        return predictions

## Inference

In [ ]:
inference = byT5Inference(config)

print(inference.model.generation_config) # these are configs saved during the training NOT at inference

test_dataset = inference.prepare_dataloader()

all_predictions = inference.translate(test_dataset)

SDPA not supported, falling back to default


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "decoder_start_token_id": 0,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 1,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_dict_in_generate": false,
  "target_lookbehind": 10,
  "temperature": 1.0,
  "top_k": 50,
  "top_p": 1.0,
  "typical_p": 1.0,
  "use_cache": true
}



100%|██████████| 4/4 [00:13<00:00,  3.28s/it]


# Submission

In [89]:
test_df = pd.read_csv(config["test_data_path"])

submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": all_predictions
})


submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken")

submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
submission.head()

Submission file saved successfully!


,id,translation
0,0,From the Kanesh colony to Aqil-<gap> our messe...
1,1,"On a tablet of the city of your city, and once..."
2,2,As soon as you have heard our message you have...
3,3,I entrusted a debt of our working capital to e...


In [90]:
print(submission.to_string())

   id                                                                                                                                                                                                                                                               translation
0   0                                                                                                                                                        From the Kanesh colony to Aqil-<gap> our messengers, every single and the stations: The tablet came to your city."
1   1                                                                                                        On a tablet of the city of your city, and once or twice, whoever makes the gold in the presence of my city, then the Kanesh colony will receive the Kanesh colony.
2   2  As soon as you have heard our message you have heard that he has given to the palace, be it he has not given to the palace, be it he has not given to our messenger. My dear fath